In [ ]:
#08_evaluate_explanations
# 
# This notebook computes explainability metrics over the results
# generated by notebooks 05 (LIME), 06 (SHAP), and 07 (BreakDown),
# for each combination of dataset, model, and XAI technique.
#
# Metrics computed:
#   - Fidelity: proportion of instances where the explainer
#     prediction matches the base model prediction
#   - Stability: consistency of explanations across the five
#     analyzed instances, measured as 1 - var(accuracy_expl)
#   - Complexity: proportion of features used in the explanation
#     relative to the total number of features in the dataset
#   - Interpretability score: average of fidelity and stability
#

import sys
sys.path.append("../src")
from utils import DATASETS


import pandas as pd
import numpy as np
import ast
from pathlib import Path
from sklearn.metrics import accuracy_score

results_paths = {
    "LIME": Path("../results/LIME"),
    "SHAP": Path("../results/SHAP"),
    "Breakdown": Path("../results/Breakdown")
}

metrics_path = Path("../results/metrics")
metrics_path.mkdir(parents=True, exist_ok=True)

processed_path = Path("../datasets/processed")

def fidelity(acc_model, acc_expl):
    return 1 - abs(acc_model - acc_expl)

rows_split = []
rows_summary = []

for method, base_path in results_paths.items():
    for ds in DATASETS:

        data = pd.read_csv(processed_path / f"{ds}_processed.csv")
        total_features = data.drop(columns=["defects"]).shape[1]

        dataset_path = base_path / ds
        if not dataset_path.exists():
            continue

        for model_dir in dataset_path.iterdir():

            if not model_dir.is_dir() or model_dir.name.startswith("."):
                continue

            model_name = model_dir.name

            csv_files = list(model_dir.glob("*.csv"))
            if not csv_files:
                continue

            df = pd.read_csv(csv_files[0])

            instance_col = "instance_id"

            if method == "LIME":
                pred_col = "lime_pred"
                values_col = "lime_weights"

            elif method == "SHAP":
                pred_col = "shap_pred"
                values_col = "shap_values"

            elif method == "Breakdown":
                pred_col = "breakdown_pred"
                values_col = "breakdown_values"

            else:
                continue

            if pred_col not in df.columns or values_col not in df.columns:
                continue

            fidelities = []
            accuracies_expl = []
            complexities = []

            for instance_id in df[instance_col].unique():

                df_instance = df[df[instance_col] == instance_id]

                method_values = df_instance[values_col].iloc[0]

                if isinstance(method_values, str):
                    method_values = ast.literal_eval(method_values)

                if method == "Breakdown":
                    method_values = method_values[1:-1]

                acc_model = accuracy_score(
                    [df_instance["true_label"].iloc[0]],
                    [df_instance["model_pred"].iloc[0]]
                )

                acc_expl = accuracy_score(
                    [df_instance["true_label"].iloc[0]],
                    [df_instance[pred_col].iloc[0]]
                )

                f = fidelity(acc_model, acc_expl)
                comp = len(method_values) / total_features

                fidelities.append(f)
                accuracies_expl.append(acc_expl)
                complexities.append(comp)

                rows_split.append({
                    "dataset": ds,
                    "model": model_name,
                    "method": method,
                    "instance_id": instance_id,
                    "fidelity": f
                })

            fidelity_mean = np.mean(fidelities)
            stability = 1 - np.var(accuracies_expl)
            complexity = np.mean(complexities)

            interpretability = (
                (fidelity_mean + stability)/2
            )

            rows_summary.append({
                "dataset": ds,
                "model": model_name,
                "method": method,  
                "fidelity": fidelity_mean,
                "stability": stability,
                "complexity": complexity,
                "interpretability": interpretability
            })

df_split = pd.DataFrame(rows_split)
df_summary = pd.DataFrame(rows_summary)

df_split.to_csv(metrics_path / "fidelity_by_instance.csv", index=False)
df_summary.to_csv(metrics_path / "metrics_summary.csv", index=False)

display(df_split)
display(df_summary)


In [ ]:
# Computational Overhead Summary

import pandas as pd
from pathlib import Path

results_root = Path("../results")
metrics_path = Path("../results/metrics")
metrics_path.mkdir(parents=True, exist_ok=True)

train_csvs = list(results_root.glob("*_models/*_training_times.csv"))
df_train = pd.concat(
    [pd.read_csv(p) for p in train_csvs], ignore_index=True
) if train_csvs else pd.DataFrame(columns=["Dataset", "Model", "Train_Time_Mean_s"])

lime_csvs      = list(results_root.glob("LIME/*/*/lime_timing.csv"))
shap_csvs      = list(results_root.glob("SHAP/*/*/shap_timing.csv"))
breakdown_csvs = list(results_root.glob("Breakdown/*/*/breakdown_timing.csv"))

dfs = []

for p in lime_csvs + shap_csvs + breakdown_csvs:
    df = pd.read_csv(p)

    if "Dataset" not in df.columns:
        parts = p.parts
        df["Dataset"] = parts[-3]
        df["Model"]   = parts[-2]
    dfs.append(df)

df_explain = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

if not df_explain.empty:
    pivot = df_explain.pivot_table(
        index=["Dataset", "Model"],
        columns="Method",
        values="Explain_Time_Mean_s",
        aggfunc="mean"
    ).reset_index()

    mem_pivot = df_explain.groupby(["Dataset", "Model"])["Mem_Delta_MB"].max().reset_index()
    mem_pivot.rename(columns={"Mem_Delta_MB": "Peak_Memory_MB"}, inplace=True)
    pivot = pivot.merge(mem_pivot, on=["Dataset", "Model"], how="left")
else:
    pivot = pd.DataFrame(columns=["Dataset", "Model", "SHAP", "LIME", "Breakdown", "Peak_Memory_MB"])


if not df_train.empty:
    df_train_agg = df_train.groupby(["Dataset", "Model"])["Train_Time_Mean_s"].mean().reset_index()
    df_train_agg.rename(columns={"Train_Time_Mean_s": "Training_Time_s"}, inplace=True)
    final = df_train_agg.merge(pivot, on=["Dataset", "Model"], how="left")
else:
    final = pivot.copy()
    final["Training_Time_s"] = float("nan")

for col in ["SHAP", "LIME", "Breakdown", "Peak_Memory_MB"]:
    if col not in final.columns:
        final[col] = float("nan")


final = final[["Dataset", "Model", "Training_Time_s", "SHAP", "LIME", "Breakdown", "Peak_Memory_MB"]]
final.columns = ["Dataset", "Model", "Training Time (s)", "SHAP (s)", "LIME (s)", "Breakdown (s)", "Peak Memory (MB)"]


for col in final.columns[2:]:
    final[col] = final[col].round(4)


explainable_models = ["Decision_Tree", "Logistic_Regression", "Naive_Bayes"]
xai_cols = ["SHAP (s)", "LIME (s)", "Breakdown (s)", "Peak Memory (MB)"]
mask = final["Model"].isin(explainable_models)
final.loc[mask, xai_cols] = "-"

model_order = ["Decision_Tree","Logistic_Regression","Naive_Bayes",
               "Random_Forest","Gradient_Boosting","SVM","MLP"]
final["_order"] = final["Model"].map({m: i for i, m in enumerate(model_order)})
final = final.sort_values(["Dataset", "_order"]).drop(columns="_order").reset_index(drop=True)

final.to_csv(metrics_path / "computational_overhead.csv", index=False)
print("Saved: results/metrics/computational_overhead.csv\n")
display(final)

